# SoftMoE-Retrieval — Train RIÊNG từng bộ DML
Hybrid CNN+ViT · Multi-scale TokenLearner · Soft MoE · RouteRank.

Mỗi bộ (CUB / Cars / In-Shop) = **một model riêng**, giao thức zero-shot chuẩn.

## 1. Cài đặt

In [ ]:
# !pip install -r ../requirements.txt
import torch; print('CUDA:', torch.cuda.is_available())

## 2. Đường dẫn data (chỉnh theo máy)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # project root
from config import HCFG
HCFG.data_roots = {'cub':'../data/CUB_200_2011','cars':'../data/cars196','inshop':'../data/inshop'}
HCFG.epochs = 60
print(HCFG.data_roots)

## 3. Smoke-test shape (tải DINOv2 + ConvNeXt)

In [ ]:
from models.hybrid_encoder import HybridEncoder
from models.hyms_route import HyMSRoute
dev='cuda' if torch.cuda.is_available() else 'cpu'
enc=HybridEncoder(HCFG.vit_name,HCFG.cnn_name,dev); enc.freeze_all()
m=HyMSRoute(enc,HCFG).to(dev)
z,rho,C=m(torch.randn(2,3,224,224).to(dev))
print('z',z.shape,'rho',rho.shape,'C',C.shape)
print('head params:', sum(p.numel() for p in m.head_parameters() if p.requires_grad))

## 4. Train một bộ

In [ ]:
!cd .. && python train.py --dataset cub --seed 42

## 5. Đa seed (mean±std cho ACCV)

In [ ]:
for s in [0,1,2]:
    !cd .. && python train.py --dataset cub --seed {s}

## 6. Eval lại từ checkpoint (base vs RouteRank)

In [ ]:
from data.dml_dataset import get_dml_loaders
from eval.routerank import evaluate_self
L=get_dml_loaders('cub',HCFG)
ck=torch.load('../results/checkpoints/best_hyms_cub_seed42.pt',map_location=dev)
m.load_state_dict(ck['model'])
r=evaluate_self(m,L['test'],dev,HCFG)
print('BASE     :',r['base']); print('RouteRank:',r['routerank'])

## Ablation (qua config)
`HCFG.cnn_stages=[1,2,3]` bỏ s1 · `HCFG.lambda_route=0` bỏ L_route · `HCFG.rr_beta=0` tắt RouteRank.